In [ ]:
def initialize_chunks(chunks_dict, num_chunks_1, num_chunks_2, L):
    """
    Initialize D_chunks and L_chunks for the first row and first column.
    Uses flexible data structure to accommodate non-square boundary chunks.
    
    Parameters:
    -----------
    chunks_dict : dict
        Dictionary containing chunk data (including 'hop' for each chunk)
    num_chunks_1, num_chunks_2 : int, int
        Number of chunks in each dimension
    L : int
        Chunk size
        
    Returns:
    --------
    D_chunks, L_chunks : list of list of dicts of lists
        Indexed as [chunk_row][chunk_col][edge_type][position]
        Where position is a Python list that can vary in length
    """
    # Initialize as nested lists with dicts for edge types
    D_chunks = [[{0: [], 1: []} for _ in range(num_chunks_2)] for _ in range(num_chunks_1)]
    L_chunks = [[{0: [], 1: []} for _ in range(num_chunks_2)] for _ in range(num_chunks_1)]
    
    # Helper function to get actual edge length for a chunk
    def get_edge_length(chunk_shape, edge_type):
        """Returns the actual length of an edge for a given chunk shape."""
        if edge_type == 0:  # top edge
            return chunk_shape[1]  # width
        else:  # right edge (edge_type == 1)
            return chunk_shape[0]  # height
    
    # Initialize chunk (0, 0)
    D_single_00 = chunks_dict[(0, 0)]['D']
    S_single_00 = chunks_dict[(0, 0)]['S']
    
    for edge_type in range(2):
        edge_len = get_edge_length(D_single_00.shape, edge_type)
        # Initialize lists with inf values
        D_chunks[0][0][edge_type] = [np.inf] * edge_len
        L_chunks[0][0][edge_type] = [np.inf] * edge_len
        
        for position in range(edge_len):
            local_row, local_col = convert_edge_to_local(edge_type, position, D_single_00.shape)
            
            if local_row < D_single_00.shape[0] and local_col < D_single_00.shape[1]:
                start_row, start_col = get_starting_point(S_single_00, local_row, local_col)
                
                if is_on_bottom_edge(start_row, start_col, D_single_00.shape) or \
                   is_on_left_edge(start_row, start_col, D_single_00.shape):
                    D_chunks[0][0][edge_type][position] = D_single_00[local_row, local_col]
                    path_length = abs(local_row - start_row) + abs(local_col - start_col)
                    L_chunks[0][0][edge_type][position] = path_length
    
    # Initialize first row: chunks (0, j) for j = 1, 2, ...
    for j in range(1, num_chunks_2):
        if (0, j) not in chunks_dict:
            continue
            
        D_single = chunks_dict[(0, j)]['D']
        S_single = chunks_dict[(0, j)]['S'] 
        C_chunk = chunks_dict[(0, j)]['C'] if 'C' in chunks_dict[(0, j)] else None
        
        for edge_type in range(2):
            edge_len = get_edge_length(D_single.shape, edge_type)
            D_chunks[0][j][edge_type] = [np.inf] * edge_len
            L_chunks[0][j][edge_type] = [np.inf] * edge_len
            
            for position in range(edge_len):
                local_row, local_col = convert_edge_to_local(edge_type, position, D_single.shape)
                
                if local_row >= D_single.shape[0] or local_col >= D_single.shape[1]:
                    continue
                
                start_row, start_col = get_starting_point(S_single, local_row, local_col)
                
                # Case 1: Valid start on bottom edge
                if is_on_bottom_edge(start_row, start_col, D_single.shape):
                    D_chunks[0][j][edge_type][position] = D_single[local_row, local_col]
                    path_length = abs(local_row - start_row) + abs(local_col - start_col)
                    L_chunks[0][j][edge_type][position] = path_length
                
                # Case 2: Start on left edge - need cost from previous chunk
                elif is_on_left_edge(start_row, start_col, D_single.shape):
                    # Get global coordinates of starting point
                    global_start_row, global_start_col = convert_local_to_global(
                        0, j, start_row, start_col, chunks_dict
                    )
                    
                    # Map to previous chunk (0, j-1)
                    prev_edge_type, prev_position = global_to_prev_chunk_edge(
                        global_start_row, global_start_col, 0, j - 1, chunks_dict, L
                    )
                    
                    # Check if prev_position is within bounds of previous chunk's edge
                    prev_edge_len = len(D_chunks[0][j-1][prev_edge_type])
                    if prev_position >= prev_edge_len:
                        continue
                    
                    # Get cost from previous chunk
                    prev_cost = D_chunks[0][j-1][prev_edge_type][prev_position]
                    
                    if np.isfinite(prev_cost):  # Only if valid path exists
                        # Subtract overlapping cell cost from CURRENT chunk to avoid double counting
                        overlap_cost = C_chunk[start_row, 0] if C_chunk is not None else 0
                        
                        D_chunks[0][j][edge_type][position] = D_single[local_row, local_col] + prev_cost - overlap_cost 
                        # Path length includes previous chunk
                        prev_length = L_chunks[0][j-1][prev_edge_type][prev_position]
                        curr_length = abs(local_row - start_row) + abs(local_col - start_col)
                        L_chunks[0][j][edge_type][position] = prev_length + curr_length
    
    # Initialize first column: chunks (i, 0) for i = 1, 2, ...
    for i in range(1, num_chunks_1):
        if (i, 0) not in chunks_dict:
            continue
            
        D_single = chunks_dict[(i, 0)]['D']
        S_single = chunks_dict[(i, 0)]['S']
        C_chunk = chunks_dict[(i, 0)]['C'] if 'C' in chunks_dict[(i, 0)] else None
        
        for edge_type in range(2):
            edge_len = get_edge_length(D_single.shape, edge_type)
            D_chunks[i][0][edge_type] = [np.inf] * edge_len
            L_chunks[i][0][edge_type] = [np.inf] * edge_len
            
            for position in range(edge_len):
                local_row, local_col = convert_edge_to_local(edge_type, position, D_single.shape)
                
                if local_row >= D_single.shape[0] or local_col >= D_single.shape[1]:
                    continue
                
                start_row, start_col = get_starting_point(S_single, local_row, local_col)
                
                # Case 1: Valid start on left edge
                if is_on_left_edge(start_row, start_col, D_single.shape):
                    D_chunks[i][0][edge_type][position] = D_single[local_row, local_col]
                    path_length = abs(local_row - start_row) + abs(local_col - start_col)
                    L_chunks[i][0][edge_type][position] = path_length
                
                # Case 2: Start on bottom edge - need cost from previous chunk
                elif is_on_bottom_edge(start_row, start_col, D_single.shape):
                    # Get global coordinates of starting point
                    global_start_row, global_start_col = convert_local_to_global(
                        i, 0, start_row, start_col, chunks_dict
                    )
                    
                    # Map to previous chunk (i-1, 0)
                    prev_edge_type, prev_position = global_to_prev_chunk_edge(
                        global_start_row, global_start_col, i - 1, 0, chunks_dict, L
                    )
                    
                    # Check if prev_position is within bounds of previous chunk's edge
                    prev_edge_len = len(D_chunks[i-1][0][prev_edge_type])
                    if prev_position >= prev_edge_len:
                        continue
                    
                    # Get cost from previous chunk
                    prev_cost = D_chunks[i-1][0][prev_edge_type][prev_position]
                    
                    if np.isfinite(prev_cost):  # Only if valid path exists
                        # Subtract overlapping cell cost from CURRENT chunk
                        overlap_cost = C_chunk[0, start_col] if C_chunk is not None else 0
                        
                        D_chunks[i][0][edge_type][position] = D_single[local_row, local_col] + prev_cost - overlap_cost
                        
                        # Path length includes previous chunk
                        prev_length = L_chunks[i-1][0][prev_edge_type][prev_position]
                        curr_length = abs(local_row - start_row) + abs(local_col - start_col)
                        L_chunks[i][0][edge_type][position] = prev_length + curr_length
    
    return D_chunks, L_chunks


def dp_fill_chunks(chunks_dict, D_chunks, L_chunks, num_chunks_1, num_chunks_2, L):
    """
    Fill in D_chunks and L_chunks for all interior chunks using dynamic programming.
    Uses flexible hop sizes from chunks_dict.
    
    Parameters:
    -----------
    chunks_dict : dict
        Dictionary containing chunk data (including flexible 'hop' per chunk)
    D_chunks, L_chunks : list of list of dicts of lists
        Chunk-level cost and length tensors (partially filled)
    num_chunks_1, num_chunks_2 : int, int
        Number of chunks in each dimension
    L : int
        Standard chunk size (for reference)
    """
    # plot_normalized_global_edge_cost(D_chunks, L_chunks, num_chunks_1, num_chunks_2)
    # Helper function to get actual edge length for a chunk
    def get_edge_length(chunk_shape, edge_type):
        """Returns the actual length of an edge for a given chunk shape."""
        if edge_type == 0:  # top edge
            return chunk_shape[1]  # width
        else:  # right edge (edge_type == 1)
            return chunk_shape[0]  # height
    
    # Process chunks row by row
    for i in range(num_chunks_1):
        for j in range(num_chunks_2):
            # Skip first row and first column (already initialized)
            if i == 0 or j == 0:
                continue
            
            D_single = chunks_dict[(i, j)]['D']
            S_single = chunks_dict[(i, j)]['S']
            C_chunk = chunks_dict[(i, j)]['C'] if 'C' in chunks_dict[(i, j)] else None
            
            for edge_type in range(2):
                edge_len = get_edge_length(D_single.shape, edge_type)
                # Initialize lists for this chunk if not already done
                D_chunks[i][j][edge_type] = [np.inf] * edge_len
                L_chunks[i][j][edge_type] = [np.inf] * edge_len
                
                for position in range(edge_len):
                    
                        
                        # plot_normalized_global_edge_cost(D_chunks, L_chunks, num_chunks_1, num_chunks_2)
                    local_row, local_col = convert_edge_to_local(edge_type, position, D_single.shape)
                    
                    if local_row >= D_single.shape[0] or local_col >= D_single.shape[1]:
                        continue
                    
                    # Get starting point for this path
                    start_row, start_col = get_starting_point(S_single, local_row, local_col)
                    
                    # Determine which previous chunk this path came from
                    if is_on_bottom_edge(start_row, start_col, D_single.shape):
                        # Path came from chunk above (i-1, j)
                        prev_i, prev_j = i - 1, j
                    elif is_on_left_edge(start_row, start_col, D_single.shape):
                        # Path came from chunk to the left (i, j-1)
                        prev_i, prev_j = i, j - 1
                    else:
                        # Path started within this chunk
                        D_chunks[i][j][edge_type][position] = D_single[local_row, local_col]
                        path_length = abs(local_row - start_row) + abs(local_col - start_col)
                        L_chunks[i][j][edge_type][position] = path_length
                        continue
                    
                    # Get global coordinates of starting point
                    global_start_row, global_start_col = convert_local_to_global(
                        i, j, start_row, start_col, chunks_dict
                    )
                    
                    # Map to edge position in previous chunk
                    prev_edge_type, prev_position = global_to_prev_chunk_edge(
                        global_start_row, global_start_col, prev_i, prev_j, chunks_dict, L
                    )
                    
                    # Check if prev_position is within bounds of previous chunk's edge
                    prev_edge_len = len(D_chunks[prev_i][prev_j][prev_edge_type])
                    if prev_position >= prev_edge_len:
                        continue
                    
                    # Get accumulated cost and length from previous chunk
                    prev_cost = D_chunks[prev_i][prev_j][prev_edge_type][prev_position]
                    prev_length = L_chunks[prev_i][prev_j][prev_edge_type][prev_position]
                    
                    # Only proceed if valid path exists in previous chunk
                    if not np.isfinite(prev_cost):
                        continue
                    
                    # Current chunk contribution (subtract first cell to avoid double counting)
                    if C_chunk is not None:
                        first_cell_cost = C_chunk[start_row, start_col]
                    else:
                        first_cell_cost = 0
                    
                    curr_cost_contribution = D_single[local_row, local_col] - first_cell_cost
                    curr_length = abs(local_row - start_row) + abs(local_col - start_col)
                    
                    # Propagate forward
                    
                    D_chunks[i][j][edge_type][position] = prev_cost + curr_cost_contribution
                    if position%4000==0:
                        print(D_chunks[i][j][edge_type][position])
                        print(i,j,edge_type,position)
                    L_chunks[i][j][edge_type][position] = prev_length + curr_length
    
    return D_chunks, L_chunks